# 09c - Auditoría y validación secundaria en `dev`

## Objetivo
Reproducir el paquete mínimo de auditoría secundaria previo a la apertura de `test`, usando únicamente artefactos ya congelados en `dev`.

Esta etapa no reabre selección de modelo, no modifica la ontología, no cambia la tarea binaria y no busca un nuevo mejor modelo.

## Entradas
- `data/splits/dataset_base.csv`
- `data/dataset_with_clinical_signal_flag.csv`
- `data/dataset_denoised.csv`
- `data/splits/train_denoised.csv`
- `data/splits/dev_denoised.csv`
- `data/outputs/cierre_modelos_dev_20260401_114409/decision_modelo_final.json`
- predicciones y modelo del XGB final congelado en `dev`
- baselines `TF-IDF`, `BETO` y `ROBERTA_CLINICAL`

## Salidas
- `data/outputs/auditoria_final_caseC_validacion_secundaria/`
- `resumen_auditoria_validacion_secundaria_dev.json`
- métricas por nota, paciente ponderado y paciente agregado
- AP/ROC-AUC de ansiedad
- sensibilidad con `sample_weight`
- trazabilidad de Caso C
- SHAP global/local por familias de variables

## Notebook anterior
- `09b_cierre_modelos_dev.ipynb` y `09_analisis_errores_hibrido.ipynb`

## Notebook siguiente
- futura auditoría de `test`, todavía no ejecutada.

## Herramientas
- `pandas`, `scikit-learn`, `xgboost`, `shap`, `joblib`.
- Script backend: `scripts/audit/generar_auditoria_validacion_secundaria_dev.py`.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display, Markdown

def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / 'data').exists() and (p / 'scripts').exists() and (p / 'notebooks').exists():
            return p
    raise RuntimeError('No se pudo resolver la raíz del repositorio')

REPO = find_repo_root(Path.cwd())
OUT_DIR = REPO / 'data' / 'outputs' / 'auditoria_final_caseC_validacion_secundaria'
SCRIPT = REPO / 'scripts' / 'audit' / 'generar_auditoria_validacion_secundaria_dev.py'

print('REPO:', REPO)
print('SCRIPT:', SCRIPT)
print('OUT_DIR:', OUT_DIR)


## Ejecutar auditoría reproducible

La celda siguiente regenera todos los artefactos de la auditoría secundaria. Debe ejecutarse con el entorno Python que tenga `xgboost` y `shap` instalados.

In [ ]:
cmd = [sys.executable, str(SCRIPT)]
result = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'Fallo la auditoría secundaria: exit={result.returncode}')


## Resumen de corrida

In [ ]:
summary_path = OUT_DIR / 'resumen_auditoria_validacion_secundaria_dev.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))

display(Markdown(f"""
**Modelo final:** `{summary['modelo_final']}`  
**Train run:** `{Path(summary['train_dir']).name}`  
**Features:** `{Path(summary['feature_path']).name}`  
**Caso C:** row_id `{summary['case_c']['row_id']}`, veredicto `{summary['case_c']['veredicto']}`  
**Reproducción XGB sin pesos:** `{summary['sample_weight']['unweighted_reproduction_mismatches_vs_frozen']}` discrepancias  
**Estado test:** auditoría formal pendiente = `{summary['checks_documentales']['test_auditoria_formal_pendiente']}`
"""))


## Métricas por nota, paciente ponderado y paciente agregado

In [ ]:
metrics_3 = pd.read_csv(OUT_DIR / 'metricas_tres_niveles_dev.csv')
display(metrics_3)


## Ansiedad: ROC-AUC, AP/PR-AUC y umbral

In [ ]:
auc_ap = pd.read_csv(OUT_DIR / 'metricas_auc_ap_ansiedad_dev.csv')
threshold_summary = json.loads((OUT_DIR / 'threshold_sweep_hibrido_dev_resumen.json').read_text(encoding='utf-8'))
display(auc_ap)
display(pd.DataFrame([
    {'criterio': 'umbral_0.50', **threshold_summary['threshold_050_row']},
    {'criterio': 'best_f1_ansiedad', **threshold_summary['best_by_f1_ansiedad']},
    {'criterio': 'best_macro_f1', **threshold_summary['best_by_macro_f1']},
]))


## Sensibilidad con `sample_weight`

In [ ]:
sample_weight_metrics = pd.read_csv(OUT_DIR / 'metricas_xgb_sample_weight_comparacion.csv')
display(sample_weight_metrics)


## Caso C

In [ ]:
case_c = json.loads((OUT_DIR / 'case_c_trace.json').read_text(encoding='utf-8'))
display(pd.DataFrame([{k: v for k, v in case_c.items() if k not in {'texto_original_raw_row', 'entities_core_reconstructed', 'prediction'}}]))
display(pd.DataFrame(case_c['entities_core_reconstructed']))
display(pd.DataFrame([case_c['prediction']]))


## Denoising y demografía descriptiva

In [ ]:
display(pd.read_csv(OUT_DIR / 'resumen_denoising_retencion.csv'))
display(pd.read_csv(OUT_DIR / 'metricas_subgrupos_dev.csv'))


## SHAP por familias

In [ ]:
shap_families = pd.read_csv(OUT_DIR / 'shap_global_familias.csv')
display(shap_families.sort_values(['class_name', 'sum_mean_abs_shap'], ascending=[True, False]))
display(pd.read_csv(OUT_DIR / 'shap_local_appendix_cases_summary.csv'))


## Lectura metodológica

- Este notebook es auditoría secundaria en `dev`, no una nueva selección.
- `sample_weight` se reporta como sensibilidad negativa, no como nuevo cierre.
- AP/PR-AUC de ansiedad debe leerse como métrica prioritaria para la clase débil.
- SHAP se interpreta por familias (`ctx_beto_*`, `rule_*`, etc.), no dimensión por dimensión.
- La sensibilidad `test_base` y la auditoría formal de `test` quedan pendientes hasta abrir `test`.